# Inference on a single case

Runs the trained model once on one case and compares the prediction with the ground truth.

- The case comes from the **validation** split. The test split is kept untouched for the
  final evaluation (`scripts/evaluate.py --split test`).
- Prediction is done exactly as in evaluation: sliding window over the full volume,
  sigmoid per region, threshold 0.5.

In [1]:
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.dataset.dataset import get_datasets
from src.dataset.paths import get_output_dir
from src.dataset.transforms import REGIONS
from src.evaluation import add_netc, case_metrics, logits_to_masks, predict_logits
from src.model import build_unet

RUN_NAME = "unet_baseline"
CHECKPOINT = "best"     # or "last"
CASE_ID = None          # e.g. "BraTS-GLI-00008-100"; None = random case from val
SEED = 0
STATIC = {"staticPlot": True}

DEVICE = torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")
print("device:", DEVICE)

device: cpu


## 1. Load the model

In [2]:
ckpt_path = get_output_dir() / "runs" / RUN_NAME / "checkpoints" / f"{CHECKPOINT}.pt"
state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
config = state["config"]
include_rc = not config.get("no_rc", False)
patch = (config["patch"],) * 3

model = build_unet(include_rc=include_rc).to(DEVICE)
model.load_state_dict(state["model"])
model.eval()

pd.Series({
    "checkpoint": str(ckpt_path.relative_to(REPO_ROOT)) if ckpt_path.is_relative_to(REPO_ROOT) else str(ckpt_path),
    "epoch": state["epoch"],
    "val mean Dice (at save)": round(state["best_dice_mean"], 4),
    "output channels": ", ".join(REGIONS[: 4 if include_rc else 3]),
    "patch / window": patch,
}, name="value").to_frame()

,value
checkpoint,outputs/runs/unet_baseline/checkpoints/best.pt
epoch,79
val mean Dice (at save),0.7659
output channels,"ET, TC, WT, RC"
patch / window,"(128, 128, 128)"


## 2. Pick a case and run the model

In [ ]:
dataset = get_datasets(patch_size=patch, include_rc=include_rc)["val"]
case_ids = [r["case_id"] for r in dataset.data]
idx = case_ids.index(CASE_ID) if CASE_ID else random.Random(SEED).randrange(len(case_ids))
sample = dataset[idx]
case_id = sample["case_id"]

image = sample["image"]   # (4, X, Y, Z) float32, preprocessed
label = sample["label"]   # (C, X, Y, Z) region masks

t0 = time.time()
logits = predict_logits(model, image[None].to(DEVICE), patch, sw_batch_size=4, overlap=0.5,
                        amp=DEVICE.type == "cuda")

if DEVICE.type == "cuda":
    torch.cuda.synchronize()
elapsed = time.time() - t0

probs = torch.sigmoid(logits)[0].cpu()           # (C, X, Y, Z) in [0, 1]
masks = logits_to_masks(logits)[0].cpu()         # (C, X, Y, Z) binary

print(f"{case_id}: volume {tuple(image.shape[1:])}, inference {elapsed:.1f} s")

BraTS-GLI-02885-101: volume (143, 172, 141), inference 4.3 s


## 3. Metrics for this case

Same per-region metrics as `scripts/evaluate.py`. NETC (= TC minus ET) is derived from the masks.
Dice is empty when the region is absent in the ground truth; HD95 (mm) needs the region in both.

In [ ]:
names = list(REGIONS[: masks.shape[0]]) + ["NETC"]
m = case_metrics(add_netc(masks), add_netc(label), names)

pd.DataFrame({
    "Dice": [m[f"dice_{n}"] for n in names],
    "HD95 (mm)": [m[f"hd95_{n}"] for n in names],
    "voxels (truth)": [m[f"gt_voxels_{n}"] for n in names],
    "voxels (prediction)": [m[f"pred_voxels_{n}"] for n in names],
}, index=names).round(3)

/home/student2/miniconda3/envs/brats/lib/python3.13/site-packages/monai/utils/deprecate_utils.py:220: FutureWarning: monai.metrics.utils get_mask_edges:always_return_as_numpy: Argument `always_return_as_numpy` has been deprecated since version 1.5.0. It will be removed in version 1.7.0. The option is removed and the return type will always be equal to the input type.
  warn_deprecated(argname, msg, warning_category)


,Dice,HD95 (mm),voxels (truth),voxels (prediction)
ET,0.858,4.774,3477,3913
TC,0.850,4.690,3477,4021
WT,0.807,3.000,11423,14395
RC,0.503,6.083,3517,1252
NETC,NaN,NaN,0,163


## 4. Regions back to labels

For display, the region masks are turned back into one label per voxel, using the nesting
ET $\subset$ TC $\subset$ WT (innermost region wins): ET $\rightarrow$ 3, TC without ET $\rightarrow$ 1 (NETC),
WT without TC $\rightarrow$ 2 (SNFH), otherwise RC $\rightarrow$ 4, else 0.

In [6]:
LABEL_NAMES = {1: "NETC", 2: "SNFH", 3: "ET", 4: "RC"}
LABEL_COLOURS = {1: "#1f77b4", 2: "#2ca02c", 3: "#d62728", 4: "#ff7f0e"}


def regions_to_labels(regions: torch.Tensor) -> np.ndarray:
    r = regions.bool().numpy()
    et, tc, wt = r[0], r[1], r[2]
    lab = np.zeros(et.shape, dtype=np.uint8)
    if r.shape[0] > 3:
        lab[r[3]] = 4
    lab[wt] = 2
    lab[tc] = 1
    lab[et] = 3
    return lab


gt_lab = regions_to_labels(label)
pred_lab = regions_to_labels(masks)
t1c = image[1].numpy()
print("labels present (truth):     ", sorted(LABEL_NAMES[v] for v in np.unique(gt_lab) if v))
print("labels present (prediction):", sorted(LABEL_NAMES[v] for v in np.unique(pred_lab) if v))

labels present (truth):      ['ET', 'RC', 'SNFH']
labels present (prediction): ['ET', 'NETC', 'RC', 'SNFH']


## 5. Truth and prediction on three planes

Slices through the centre of the tumour (whole tumour, or the volume centre if there is none).

In [ ]:
def centre(mask: np.ndarray) -> tuple[int, int, int]:
    idx = np.argwhere(mask)
    return tuple(int(v) for v in (idx.mean(0) if len(idx) else np.array(mask.shape) / 2))


def cut(vol, plane, pos):
    x, y, z = pos
    s = {"axial": vol[:, :, z], "coronal": vol[:, y, :], "sagittal": vol[x, :, :]}[plane]
    return np.rot90(s)  # anterior / superior up


def label_overlay(lab2d):
    z = np.where(lab2d > 0, lab2d, np.nan).astype(float)
    scale = []
    for v in range(1, 5):
        lo, hi = (v - 1) / 4, v / 4
        scale += [[lo, LABEL_COLOURS[v]], [hi, LABEL_COLOURS[v]]]
    return go.Heatmap(z=z, zmin=0.5, zmax=4.5, colorscale=scale, showscale=False, opacity=0.55,
                      hoverinfo="skip")


pos = centre(gt_lab > 0)
planes = ["axial", "coronal", "sagittal"]
fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.02, vertical_spacing=0.06,
                    row_titles=["truth", "prediction"], column_titles=planes)
            
for row, lab in enumerate((gt_lab, pred_lab), start=1):
    for col, plane in enumerate(planes, start=1):
        fig.add_trace(go.Heatmap(z=cut(t1c, plane, pos), colorscale="gray", showscale=False,
                                 hoverinfo="skip"), row, col)
        fig.add_trace(label_overlay(cut(lab, plane, pos)), row, col)
    
for v, name in LABEL_NAMES.items():
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=name,
                             marker=dict(symbol="square", size=14, color=LABEL_COLOURS[v])))

fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
for i in range(1, 7):
    sx = "" if i == 1 else str(i)
    fig.layout[f"yaxis{sx}"].update(scaleanchor=f"x{sx}")
fig.update_layout(width=1000, height=720, plot_bgcolor="black", margin=dict(t=60, b=40, l=40, r=10),
                  title=dict(text=f"<b>{case_id}</b> — t1c with labels, slices through {pos}", x=0.5),
                  legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.03, yanchor="top"))
fig.show(config=STATIC)

## 6. Probability maps

Sigmoid output per region on the axial slice, before thresholding. Values near 0.5 show where
the model is unsure; the contour is the ground truth.

In [8]:
z = pos[2]
n = probs.shape[0]
fig = make_subplots(rows=1, cols=n, subplot_titles=REGIONS[:n], horizontal_spacing=0.02)
for c in range(n):
    fig.add_trace(go.Heatmap(z=cut(probs[c].numpy(), "axial", pos), zmin=0, zmax=1, colorscale="Viridis",
                             showscale=c == n - 1, colorbar=dict(title="p", len=0.8), hoverinfo="skip"), 1, c + 1)
    fig.add_trace(go.Contour(z=cut(label[c].numpy(), "axial", pos), contours=dict(start=0.5, end=0.5,
                             coloring="none"), line=dict(color="white", width=2), showscale=False,
                             hoverinfo="skip"), 1, c + 1)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
for i in range(1, n + 1):
    sx = "" if i == 1 else str(i)
    fig.layout[f"yaxis{sx}"].update(scaleanchor=f"x{sx}")
fig.update_layout(width=260 * n + 80, height=340, margin=dict(t=40, b=10, l=10, r=10))
fig.show(config=STATIC)

## 7. Errors per region

True positives, false positives and false negatives on the axial slice, per region.

In [9]:
TP, FP, FN, TN = "#4c9a6a", "#c56d6d", "#5b7fb5", "#000000"
err_scale = [[0, TN], [0.25, TN], [0.25, FP], [0.5, FP], [0.5, FN], [0.75, FN], [0.75, TP], [1, TP]]

fig = make_subplots(rows=1, cols=n, subplot_titles=REGIONS[:n], horizontal_spacing=0.02)
for c in range(n):
    p, t = masks[c].bool().numpy(), label[c].bool().numpy()
    cat = np.zeros(p.shape, dtype=int)
    cat[p & ~t], cat[~p & t], cat[p & t] = 1, 2, 3
    fig.add_trace(go.Heatmap(z=cut(cat, "axial", pos), zmin=-0.5, zmax=3.5, colorscale=err_scale,
                             showscale=False, hoverinfo="skip"), 1, c + 1)
for name, colour in (("true positive", TP), ("false positive", FP), ("false negative", FN)):
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=name,
                             marker=dict(symbol="square", size=14, color=colour)))
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
for i in range(1, n + 1):
    sx = "" if i == 1 else str(i)
    fig.layout[f"yaxis{sx}"].update(scaleanchor=f"x{sx}")
fig.update_layout(width=260 * n, height=380, margin=dict(t=40, b=40, l=10, r=10),
                  legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.05, yanchor="top"))
fig.show(config=STATIC)

## 8. 3D reconstruction

In [ ]:
from scipy.ndimage import gaussian_filter
from skimage.measure import marching_cubes

# per-label look: outer regions transparent so inner ones stay visible
LABEL_OPACITY = {2: 0.25, 4: 0.45, 1: 0.8, 3: 1.0}


def mesh(mask: np.ndarray, step: int = 1, sigma: float = 1.0):
    """Smoothed surface of a binary mask as (vertices, faces); None if too small."""
    if mask.sum() < 20:
        return None
        
    padded = np.pad(mask.astype(np.float32), 2)  # pad: close surfaces at edges
    vol = gaussian_filter(padded, sigma)

    if vol.max() <= 0.5:   # thin or scattered structure vanished after smoothing
        vol = padded        # fall back to the raw mask (values 0 and 1)

    verts, faces, _, _ = marching_cubes(vol, level=0.5, step_size=step)
    return verts - 2, faces


brain = (np.asarray(image) != 0).any(axis=0)
brain_mesh = mesh(brain, step=3, sigma=2.0)

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
                    subplot_titles=("ground truth", "prediction"), horizontal_spacing=0.01)
for col, lab in enumerate((gt_lab, pred_lab), start=1):
    if brain_mesh is not None:
        v, f = brain_mesh
        fig.add_trace(go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2], i=f[:, 0], j=f[:, 1], k=f[:, 2],
                                color="lightgrey", opacity=0.08, hoverinfo="skip",
                                name="brain", legendgroup="brain", showlegend=col == 1), 1, col)
    for value in (2, 4, 1, 3):  # outer first, inner last
        m = mesh(lab == value)
        if m is None:
            continue
        v, f = m
        fig.add_trace(go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2], i=f[:, 0], j=f[:, 1], k=f[:, 2],
                                color=LABEL_COLOURS[value], opacity=LABEL_OPACITY[value],
                                flatshading=False, lighting=dict(ambient=0.5, diffuse=0.8, specular=0.2),
                                name=LABEL_NAMES[value], legendgroup=LABEL_NAMES[value],
                                showlegend=col == 1, hoverinfo="name"), 1, col)

camera = dict(eye=dict(x=1.6, y=-1.6, z=0.9))  # same starting view on both sides

scene = dict(aspectmode="data", camera=camera,
             xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False))
             
fig.update_layout(scene=scene, scene2=scene, width=1100, height=560,
                  margin=dict(t=50, b=10, l=10, r=10),
                  title=dict(text=f"<b>{case_id}</b> — 3D reconstruction (drag to rotate, scroll to zoom)", x=0.5),
                  legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.02, yanchor="top"))
fig.show()